# **CrisperWhisper2.0 for Audio Transcription**

Welcome! This notebook provides an easy-to-use interface for transcribing audio using [CrisperWhisper](https://github.com/nyrahealth/CrisperWhisper) — a state-of-the-art speech-to-text model designed for both extreme precision and clean readability.


* **Two Transcription Modes:**
  * **Verbatim:** Captures exactly what was said, including every filler word ("um", "uh"), stutter, repetition, and false start.
  * **Intended:** Outputs a clean, highly readable version of the speech by stripping out disfluencies.


### How to Use This Notebook

1. **Step 1 (Setup):** Run the first code cell to install the required libraries and load the model into the GPU. You only need to run this **once**.
2. **Step 2 (Transcribe):** Run the second code cell to upload your audio file (`.wav`, `.mp3`, `.m4a`), choose your transcription mode, and automatically download your transcript. You can run this cell as many times as you want for different audio files.


In [ ]:
# @markdown # 🛠️ Step 1: Setup Environment & Load Model
# @markdown Select the model size (default is CrisperWhisper 2.0 large which is slower than others, but produces transcripts with the highest quality):
model_size = "nyralabs/CrisperWhisper2.0_large" # @param ["nyralabs/CrisperWhisper2.0_large", "turbo", "medium", "small"]

# 1. Install dependencies
print("Installing dependencies (this may take a minute)...")
!pip install -q "crisperwhisper[ct2]"
!apt-get install -q -y ffmpeg
from IPython.display import clear_output

# 2. Import libraries
import os
import re
import warnings
from google.colab import files
from crisperwhisper import CrisperWhisperModel

# Suppress warnings for a cleaner output
warnings.filterwarnings("ignore")

# 3. Load the CrisperWhisper Model
print(f"\nLoading CrisperWhisper model ({model_size})...")
model = CrisperWhisperModel(model_size)
clear_output()
print("✅ Model loaded successfully! You can now run the next cell.")

In [ ]:
# @title {"single-column":true}
# @markdown # 💥 Step 2: Upload Audio & Transcribe
# @markdown Select the transcription mode:
transcription_mode = "verbatim" # @param ["verbatim", "intended"]
download_transcript=True # @param {type:"boolean"}

print("Please upload your audio file (e.g., audio.wav, .mp3, .m4a):")
# 1. Interactive File Upload
uploaded = files.upload()

if not uploaded:
    raise ValueError("No file uploaded. Please run the cell again and select a file.")

audio_filename = list(uploaded.keys())[0]
print(f"\nSuccessfully uploaded: {audio_filename}")

# 2. Run Transcription
print(f"Transcribing audio in '{transcription_mode}' mode...")
# We use the 'model' loaded in Cell 1
result = model.transcribe(audio_filename, language="en", mode=transcription_mode)

# 3. Process Text (Stitch fragments and create paragraphs)
continuous_text = result.text.strip()
sentences = re.split(r'(?<=[.!?]) +', continuous_text)
sentences = [s.strip() for s in sentences if s.strip()]

paragraphs = []
sentences_per_paragraph = 5

for i in range(0, len(sentences), sentences_per_paragraph):
    paragraph = " ".join(sentences[i:i + sentences_per_paragraph])
    paragraphs.append(paragraph)

formatted_document = "\n\n".join(paragraphs)

# 4. Save to a Text Document
base_name = os.path.splitext(audio_filename)[0]
txt_filename = f"{base_name}_{transcription_mode}_transcript.txt"

with open(txt_filename, "w", encoding="utf-8") as f:
    f.write(formatted_document)

print(f"\n✅ Transcription complete! Saved to: {txt_filename}")
if download_transcript:
    # 5. Automatic File Download
    print("Triggering download...")
    files.download(txt_filename)
    print("Also check your download folder for the transcript.")